In [ ]:
# ============================================================
# TALTECH AI PIPELINE
# NOTEBOOK 05B — DATA QUALITY AUDIT
# Portable Windows/Linux
#
# IMPORTANT:
# This notebook audits the synchronized dataset WITHOUT
# automatically deleting statistical outliers.
#
# INPUT:
# outputs/05_ml_dataset/05_synchronized_wide_all_slots.csv
#
# OUTPUT:
# outputs/05B_data_quality/
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
INPUT_FILE = (
    OUTPUT_DIR
    / "05_ml_dataset"
    / "05_synchronized_wide_all_slots.csv"
)
OUT = OUTPUT_DIR / "05B_data_quality"
OUT.mkdir(parents=True, exist_ok=True)

print("====================================")
print("NOTEBOOK 05B — DATA QUALITY AUDIT")
print("====================================")
print("Project :", PROJECT_DIR)
print("Input   :", INPUT_FILE)
print("Output  :", OUT)

assert INPUT_FILE.exists(), (
    f"\nInput file not found:\n{INPUT_FILE}\n"
    "Run Notebook 05 first."
)

# ============================================================
# 2. LOAD
# ============================================================

df = pd.read_csv(INPUT_FILE)
df["timestamp"] = pd.to_datetime(
    df["timestamp"], utc=True, errors="coerce"
)

variables = ["temperature", "relative_humidity", "co2"]

for col in variables:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nRows    :", len(df))
print("Devices :", df["uuid"].nunique())
print("Period  :", df["timestamp"].min(), "->", df["timestamp"].max())

# ============================================================
# 3. STRUCTURAL CHECKS
# ============================================================

invalid_timestamps = int(df["timestamp"].isna().sum())
duplicate_rows = int(
    df.duplicated(subset=["uuid", "timestamp"]).sum()
)

structural = pd.DataFrame([
    {
        "check": "rows",
        "value": len(df)
    },
    {
        "check": "devices",
        "value": df["uuid"].nunique()
    },
    {
        "check": "invalid_timestamps",
        "value": invalid_timestamps
    },
    {
        "check": "duplicate_uuid_timestamp",
        "value": duplicate_rows
    }
])

structural.to_csv(
    OUT / "01_structural_quality_checks.csv",
    index=False
)

# ============================================================
# 4. MISSINGNESS BY DEVICE / VARIABLE
# ============================================================

missing_rows = []

for uuid, part in df.groupby("uuid"):
    for variable in variables:
        n = len(part)
        missing = int(part[variable].isna().sum())
        available = n - missing

        missing_rows.append({
            "uuid": uuid,
            "variable": variable,
            "total_slots": n,
            "available": available,
            "missing": missing,
            "missing_pct": 100 * missing / n,
            "completeness_pct": 100 * available / n
        })

missingness = pd.DataFrame(missing_rows)
missingness.to_csv(
    OUT / "02_missingness_by_device_variable.csv",
    index=False
)

# ============================================================
# 5. DISTRIBUTION / DESCRIPTIVE STATISTICS
# ============================================================

stats_rows = []

for uuid, part in df.groupby("uuid"):
    for variable in variables:

        s = part[variable].dropna()

        if len(s) == 0:
            continue

        stats_rows.append({
            "uuid": uuid,
            "variable": variable,
            "n": len(s),
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "q01": s.quantile(0.01),
            "q05": s.quantile(0.05),
            "q25": s.quantile(0.25),
            "median": s.median(),
            "q75": s.quantile(0.75),
            "q95": s.quantile(0.95),
            "q99": s.quantile(0.99),
            "max": s.max()
        })

stats = pd.DataFrame(stats_rows)
stats.to_csv(
    OUT / "03_descriptive_statistics.csv",
    index=False
)

# ============================================================
# 6. IQR FLAGS — DIAGNOSTIC ONLY
# ============================================================
# IMPORTANT:
# IQR outliers are NOT deleted. In building sensor data,
# unusual values may correspond to genuine operating events.

iqr_rows = []

for uuid, part in df.groupby("uuid"):
    for variable in variables:

        s = part[variable].dropna()

        if len(s) == 0:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        flagged = ((s < lower) | (s > upper))

        iqr_rows.append({
            "uuid": uuid,
            "variable": variable,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_iqr_bound": lower,
            "upper_iqr_bound": upper,
            "iqr_flagged_values": int(flagged.sum()),
            "iqr_flagged_pct":
                100 * flagged.sum() / len(s)
        })

iqr_audit = pd.DataFrame(iqr_rows)
iqr_audit.to_csv(
    OUT / "04_iqr_diagnostic_flags.csv",
    index=False
)

# ============================================================
# 7. PHYSICAL PLAUSIBILITY FLAGS
# ============================================================
# Conservative diagnostic limits.
# These are screening rules, not automatic deletion rules.
#
# Temperature: -10 to 50 °C
# RH:           0 to 100 %
# CO2:          0 to 10000 ppm
#
# Values outside these broad limits are flagged for review.

physical_limits = {
    "temperature": (-10.0, 50.0),
    "relative_humidity": (0.0, 100.0),
    "co2": (0.0, 10000.0)
}

physical_rows = []
flagged_records = []

for variable, (low, high) in physical_limits.items():

    mask = (
        df[variable].notna()
        &
        (
            (df[variable] < low)
            |
            (df[variable] > high)
        )
    )

    physical_rows.append({
        "variable": variable,
        "screening_min": low,
        "screening_max": high,
        "flagged_values": int(mask.sum()),
        "flagged_pct_of_available":
            (
                100 * mask.sum() / df[variable].notna().sum()
                if df[variable].notna().sum() else np.nan
            )
    })

    if mask.any():
        tmp = df.loc[
            mask,
            ["uuid", "timestamp", variable]
        ].copy()

        tmp = tmp.rename(columns={variable: "value"})
        tmp["variable"] = variable
        tmp["screening_min"] = low
        tmp["screening_max"] = high

        flagged_records.append(tmp)

physical = pd.DataFrame(physical_rows)
physical.to_csv(
    OUT / "05_physical_plausibility_audit.csv",
    index=False
)

if flagged_records:
    physical_details = pd.concat(
        flagged_records, ignore_index=True
    )
else:
    physical_details = pd.DataFrame(
        columns=[
            "uuid", "timestamp", "value", "variable",
            "screening_min", "screening_max"
        ]
    )

physical_details.to_csv(
    OUT / "06_physical_plausibility_flagged_records.csv",
    index=False
)

# ============================================================
# 8. TEMPORAL REGULARITY
# ============================================================

temporal_rows = []

for uuid, part in df.groupby("uuid"):

    part = part.sort_values("timestamp")
    delta = part["timestamp"].diff().dropna()

    temporal_rows.append({
        "uuid": uuid,
        "rows": len(part),
        "expected_step_minutes": 15,
        "steps_equal_15min":
            int((delta == pd.Timedelta(minutes=15)).sum()),
        "steps_not_15min":
            int((delta != pd.Timedelta(minutes=15)).sum()),
        "min_step_minutes":
            delta.min().total_seconds() / 60 if len(delta) else np.nan,
        "max_step_minutes":
            delta.max().total_seconds() / 60 if len(delta) else np.nan
    })

temporal = pd.DataFrame(temporal_rows)
temporal.to_csv(
    OUT / "07_temporal_regularity.csv",
    index=False
)

# ============================================================
# 9. CONSTANT / NEAR-CONSTANT SIGNAL CHECK
# ============================================================

signal_rows = []

for uuid, part in df.groupby("uuid"):
    for variable in variables:

        s = part[variable].dropna()

        signal_rows.append({
            "uuid": uuid,
            "variable": variable,
            "available": len(s),
            "unique_values": s.nunique(),
            "variance": s.var() if len(s) > 1 else np.nan,
            "constant_signal":
                bool(s.nunique() <= 1) if len(s) else True
        })

signal = pd.DataFrame(signal_rows)
signal.to_csv(
    OUT / "08_signal_variability_check.csv",
    index=False
)

# ============================================================
# 10. SUDDEN CHANGE DIAGNOSTIC
# ============================================================
# We do not impose a scientific threshold here.
# We calculate absolute first differences and their quantiles,
# which can later support sensor-quality decisions.

change_rows = []

for uuid, part in df.groupby("uuid"):

    part = part.sort_values("timestamp")

    for variable in variables:

        delta = part[variable].diff().abs().dropna()

        if len(delta) == 0:
            continue

        change_rows.append({
            "uuid": uuid,
            "variable": variable,
            "n_differences": len(delta),
            "median_abs_change": delta.median(),
            "q95_abs_change": delta.quantile(0.95),
            "q99_abs_change": delta.quantile(0.99),
            "max_abs_change": delta.max()
        })

changes = pd.DataFrame(change_rows)
changes.to_csv(
    OUT / "09_sudden_change_diagnostics.csv",
    index=False
)

# ============================================================
# 11. OVERALL QUALITY SUMMARY
# ============================================================

summary = pd.DataFrame([
    {
        "metric": "dataset_rows",
        "value": len(df)
    },
    {
        "metric": "devices",
        "value": df["uuid"].nunique()
    },
    {
        "metric": "duplicate_uuid_timestamp",
        "value": duplicate_rows
    },
    {
        "metric": "invalid_timestamps",
        "value": invalid_timestamps
    },
    {
        "metric": "temperature_missing",
        "value": int(df["temperature"].isna().sum())
    },
    {
        "metric": "relative_humidity_missing",
        "value": int(df["relative_humidity"].isna().sum())
    },
    {
        "metric": "co2_missing",
        "value": int(df["co2"].isna().sum())
    },
    {
        "metric": "physical_plausibility_flags",
        "value": len(physical_details)
    },
    {
        "metric": "constant_device_variable_series",
        "value": int(signal["constant_signal"].sum())
    }
])

summary.to_csv(
    OUT / "10_data_quality_summary.csv",
    index=False
)

# ============================================================
# 12. CLEANING DECISION MANIFEST
# ============================================================
# This explicitly records what this notebook did and did not do.

manifest = pd.DataFrame([
    {
        "rule": "short_missing_gaps",
        "decision":
            "Already interpolated in Notebook 05 only when <= 1 hour."
    },
    {
        "rule": "long_missing_gaps",
        "decision":
            "Preserved as missing; no interpolation."
    },
    {
        "rule": "IQR_outliers",
        "decision":
            "Flagged for diagnosis only; not deleted."
    },
    {
        "rule": "physical_plausibility",
        "decision":
            "Broad screening only; flagged records require review."
    },
    {
        "rule": "duplicates",
        "decision":
            "Audited after 15-min aggregation."
    },
    {
        "rule": "scaling",
        "decision":
            "NOT performed here; fit only on training data after temporal split."
    },
    {
        "rule": "final_cleaning",
        "decision":
            "No automatic removal in Notebook 05B."
    }
])

manifest.to_csv(
    OUT / "11_cleaning_decision_manifest.csv",
    index=False
)

# ============================================================
# 13. PRINT RESULTS
# ============================================================

print("\n====================================")
print("STRUCTURAL QUALITY")
print("====================================")
print(structural.to_string(index=False))

print("\n====================================")
print("MISSINGNESS")
print("====================================")
print(missingness.to_string(index=False))

print("\n====================================")
print("PHYSICAL PLAUSIBILITY")
print("====================================")
print(physical.to_string(index=False))

print("\n====================================")
print("IQR DIAGNOSTIC")
print("====================================")
print(
    iqr_audit[
        [
            "uuid",
            "variable",
            "iqr_flagged_values",
            "iqr_flagged_pct"
        ]
    ].to_string(index=False)
)

print("\n====================================")
print("TEMPORAL REGULARITY")
print("====================================")
print(temporal.to_string(index=False))

print("\n====================================")
print("FINAL SUMMARY")
print("====================================")
print(summary.to_string(index=False))

print("\n====================================")
print("FILES CREATED")
print("====================================")
for f in sorted(OUT.iterdir()):
    print(" -", f.name)

print("\n✓ Data-quality audit completed.")
print("✓ No statistical outlier was automatically removed.")
print("✓ No scaling was performed.")
print("✓ Long missing gaps remain protected.")
